In [ ]:
!pip install scraper

### 1. Get your Google API Key

To use Google's generative AI models, you'll need an API key. You can get one from [Google AI Studio](https://aistudio.google.com/app/apikey).

Once you have your API key, add it to Colab's Secrets Manager for secure storage. Click on the '🔑' icon in the left sidebar, add a new secret, and name it `GOOGLE_API_KEY`. Then, you can access it in your notebook like this:

In [ ]:
import os
from dotenv import load_dotenv
from scraper import *
from IPython.display import Markdown, display

# Import the Python SDK
import google.generativeai as genai

# Used to securely store your API key
from google.colab import userdata

# Configure the API key
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Google Generative AI API configured successfully!")

Google Generative AI API configured successfully!


### 2. Initialize a Generative Model

Now you can initialize a Gemini model. For general purposes, `gemini-pro` is a good starting point.

In [ ]:
# Initialize a Generative Model. Using gemini-1.5-flash as gemini-pro had issues.
model = genai.GenerativeModel('gemini-2.5-flash')

print("Gemini-2.5-flash model initialized!")

Gemini-2.5-flash model initialized!


### 3. Make a Request to the Model

Here's an example of how to generate text using the initialized model:

In [ ]:
# Generate content
response = model.generate_content("Tell me a short, interesting fact about the universe.")

messages = [{"role": "user", "content": response}]

messages

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 31.06405459s.

In [ ]:
import requests

headers = {"Authorization": f"Bearer {GOOGLE_API_KEY}", "Content-Type": "application/json"}

payload = {
    "model": "gemini-2.5-flash",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

In [ ]:
response.text

### Option 1: Changing the model when using the `google.generativeai` SDK

In [ ]:
# Initialize a different Generative Model, for example, 'gemini-1.5-flash'
# You would replace 'gemini-1.5-flash' with the desired model name.
# As 'gemini-1.5-flash' resulted in a NotFound error, let's try a known working model like 'gemini-2.5-flash' or another available model.
# You can check available models using `for m in genai.list_models(): print(m.name)`
new_model_sdk = genai.GenerativeModel('gemini-2.5-flash') # Changed to a known working model

print(f"New model initialized: {new_model_sdk.model_name}")

# You can then use this new_model_sdk to generate content
response_new_sdk = new_model_sdk.generate_content("What is the capital of France?")
print(f"Response from new SDK model: {response_new_sdk.text}")

### Option 2: Changing the model when using `requests` (manual API call)

In [ ]:
# Update the 'model' key in your payload dictionary
# Replace 'gemini-1.5-flash' with the desired model name.
new_payload_requests = {
    "model": "gemini-1.5-flash",
    "messages": [
        {"role": "user", "content": "Who was the first person on the moon?"}]
}

print(f"New payload for requests: {new_payload_requests}")

# To make the actual API call, you would typically use requests.post()
# For example (assuming you have a base URL for the API):
# import requests
# api_url = "https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent"
# headers = {"Content-Type": "application/json", "x-goog-api-key": GOOGLE_API_KEY}
#
# # Replace {model_id} with the actual model name from your payload
# model_id_from_payload = new_payload_requests['model']
# specific_api_url = api_url.format(model_id=model_id_from_payload)
#
# raw_response = requests.post(specific_api_url, headers=headers, json=new_payload_requests)
# if raw_response.status_code == 200:
#     print(f"Response from new requests model: {raw_response.json()['candidates'][0]['content']['parts'][0]['text']}")
# else:
#     print(f"Error: {raw_response.status_code} - {raw_response.text}")

### Tokenizer

In [ ]:
# To count tokens for Gemini models, use the model's count_tokens method from the google.generativeai library.
# The `tiktoken` library is designed for OpenAI models and does not support Gemini models.

# Assuming 'model' variable is already initialized as a GenerativeModel for 'gemini-2.5-flash'
# from cell 51693b73: model = genai.GenerativeModel('gemini-2.5-flash')

token_count_response = model.count_tokens("Hi my name is Aditya and I like Coffee")
print(f"Token count for the text: {token_count_response.total_tokens}")

## Week 1 Project


In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
# The following import caused an ImportError because 'fetch_website_links' and 'fetch_website_contents'
# were not found in the installed 'scraper' package (version 0.1.0).
# You may need to define these functions or use a different web scraping library.
# from scraper import fetch_website_links, fetch_website_contents

In [ ]:
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_website_contents(url):
    """Fetches the content of a given URL."""
    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching content from {url}: {e}")
        return None

def fetch_website_links(url):
    """Fetches all unique links from a given URL."""
    content = fetch_website_contents(url)
    if content:
        soup = BeautifulSoup(content, 'html.parser')
        links = set()
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            # Resolve relative URLs to absolute URLs if necessary
            if not href.startswith('http') and not href.startswith('#'):
                # Simple join, more robust solution might use urllib.parse.urljoin
                if url.endswith('/') and href.startswith('/'):
                    full_url = url + href[1:]
                elif url.endswith('/') or href.startswith('/'):
                    full_url = url + href
                else:
                    full_url = url + '/' + href
                links.add(full_url)
            elif href.startswith('http'):
                links.add(href)
        return list(links)
    return []

print("Custom web scraping functions 'fetch_website_contents' and 'fetch_website_links' are now defined.")

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    # Construct the user prompt content
    user_prompt_content = get_links_user_prompt(url)

    # Combine the system instruction with the user prompt, as 'system_instruction'
    # is not a direct parameter for generate_content in this context.
    full_prompt_content = link_system_prompt + "\n\n" + user_prompt_content

    # Use the initialized genai model for content generation
    # Pass the combined prompt directly to contents.
    response = model.generate_content(
        contents=[full_prompt_content]
    )

    # The actual content of the response is in response.text
    result = response.text

    # !!! DIAGNOSTIC STEP: Print the raw response from the model !!!
    print(f"Raw model response: {result}")

    # Remove markdown code block delimiters if present
    if result.startswith('```json') and result.endswith('```'):
        result = result[len('```json'):-len('```')].strip()
    elif result.startswith('```') and result.endswith('```'): # Generic markdown block
        result = result[len('```'):-len('```')].strip()

    # Parse the JSON string from the model's response
    links = json.loads(result)
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")